In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2022_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/precos-semestrais-ca-2022-01.csv", sep=";")

In [3]:
df_2022_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2022-02.csv", sep=";")

In [4]:
df_2022 = pd.concat((df_2022_01,df_2022_02))

In [5]:
df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 907502 entries, 0 to 378077
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     907502 non-null  object 
 1   Estado - Sigla     907502 non-null  object 
 2   Municipio          907502 non-null  object 
 3   Revenda            907502 non-null  object 
 4   CNPJ da Revenda    907502 non-null  object 
 5   Nome da Rua        907502 non-null  object 
 6   Numero Rua         907113 non-null  object 
 7   Complemento        195200 non-null  object 
 8   Bairro             905168 non-null  object 
 9   Cep                907502 non-null  object 
 10  Produto            907502 non-null  object 
 11  Data da Coleta     907502 non-null  object 
 12  Valor de Venda     907502 non-null  object 
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  907502 non-null  object 
 15  Bandeira           907502 non-null  object 
dtypes: floa

In [6]:
df_2022_pe = df_2022[df_2022["Estado - Sigla"] == "PE"]

In [7]:
df_2022_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
488,NE,PE,BELO JARDIM,AUTO POSTO JURITY LTDA,04.907.385/0002-07,AVENIDA OSCAR PEREIRA DA SILVA,364,NaN,SANTO ANTÔNIO,55150-000,GASOLINA,03/01/2022,"6,58",NaN,R$ / litro,RAIZEN
489,NE,PE,BELO JARDIM,AUTO POSTO JURITY LTDA,04.907.385/0002-07,AVENIDA OSCAR PEREIRA DA SILVA,364,NaN,SANTO ANTÔNIO,55150-000,ETANOL,03/01/2022,"5,35",NaN,R$ / litro,RAIZEN
490,NE,PE,BELO JARDIM,AUTO POSTO JURITY LTDA,04.907.385/0002-07,AVENIDA OSCAR PEREIRA DA SILVA,364,NaN,SANTO ANTÔNIO,55150-000,DIESEL S10,03/01/2022,"5,28",NaN,R$ / litro,RAIZEN
491,NE,PE,BELO JARDIM,J R INACIO COMBUSTIVEIS,02.954.358/0001-89,PRACA DESEMBARGADOR JOAO PAES,SN,NaN,CENTRO,55150-170,GASOLINA,03/01/2022,"6,58",NaN,R$ / litro,ALESAT
492,NE,PE,BELO JARDIM,J R INACIO COMBUSTIVEIS,02.954.358/0001-89,PRACA DESEMBARGADOR JOAO PAES,SN,NaN,CENTRO,55150-170,ETANOL,03/01/2022,"5,39",NaN,R$ / litro,ALESAT


In [8]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Staging no BigQuery
table_id = os.getenv("BRONZE_2022")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [9]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2022_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
